# Leakage check

In [1]:
import pandas as pd

from demand.features import FEATURES, build_features
from demand.config import HORIZON
from demand.data import load_subset, clean

df = clean(load_subset())
features = build_features(df)

limit = pd.Timestamp("2017-05-01")
altered = df.copy()
altered.loc[altered.date > limit, "sales"] *= 10
altered_features = build_features(altered)

safe = features.date <= limit + pd.Timedelta(days=HORIZON)
pd.testing.assert_frame_equal(features.loc[safe, FEATURES], altered_features.loc[safe, FEATURES])
print("no leakage: features up to", (limit + pd.Timedelta(days=HORIZON)).date(), "do not depend on sales after", limit.date())

no leakage: features up to 2017-05-29 do not depend on sales after 2017-05-01


# Missing values in features

In [2]:
features[FEATURES].isna().mean().round(3).sort_values(ascending=False)

sales_lag_56          0.082
sales_lag_42          0.063
sales_lag_35          0.053
promo_rel             0.047
sales_lag_28          0.044
sales_roll_std_28     0.040
sales_roll_mean_7     0.038
sales_roll_mean_56    0.038
sales_roll_mean_28    0.038
family                0.000
store_nbr             0.000
dayofweek             0.000
day                   0.000
days_to_christmas     0.000
month                 0.000
holiday_national      0.000
holiday_local         0.000
is_event              0.000
onpromotion           0.000
dtype: float64

# Conclusions

- Strategy: one direct model for the 28-day horizon. All lags and rolling windows are shifted by at least 28 days, so every feature is known at forecast time and can be computed once on the whole dataset.
- Features: sales lags 28, 35, 42, 56 (multiples of 7, same weekday), rolling mean (7, 28, 56 days) and std (28 days) of sales shifted by 28, calendar (day of week, day, month, days to Christmas), holiday and event flags, and `onpromotion` with its relative version `promo_rel`.
- Closed days are set to NaN before lags and rolling windows, so a closed day does not become a zero in the next weeks (the seasonal naive problem from the baselines).
- Leakage check passed: changing sales after 2017-05-01 does not change the features of any day up to 2017-05-29.
- Missing values: the first 28-56 days of every series (lags and windows are not filled yet) and about 0.5% from closed days. Nothing unexpected.
- `usable_rows` drops the first 56 days (no full history) and the closed days. It is applied inside each walk-forward fold, separately to train and validation, with the history start of the whole dataset.
- Removed after the ablation in `05_models`: `is_payday` (no effect) and `oil_lag_28` (a trending proxy for time, it hurt validation: WAPE 0.126 with it, 0.112 without).